# 🔒 Day 02a: Mutex, Semaphore & Monitors

---

## 🎯 What You'll Master Today
- Mutex vs Binary Semaphore vs Counting Semaphore
- Spinlock, Monitor, Condition Variables
- Python's Lock, Semaphore, Condition, Event

---

```
╔══════════════════════════════════════════════════════════════╗
║           SYNCHRONIZATION PRIMITIVES                         ║
╠═══════════════════╦══════════════════════════════════════════╣
║  Mutex             ║  Binary lock. Only owner can unlock.   ║
║  Binary Semaphore  ║  0 or 1. Any thread can signal.        ║
║  Counting Sem.     ║  0 to N. Controls access to N units.   ║
║  Spinlock          ║  Busy-wait mutex (no sleep).           ║
║  Monitor           ║  High-level: mutex + condition vars.   ║
║  Condition Var     ║  Wait/notify inside a monitor.         ║
╚═══════════════════╩══════════════════════════════════════════╝
```

> **Key difference:** Mutex has OWNERSHIP (only locker can unlock). Semaphore has no ownership (anyone can signal).

---

In [ ]:
# ============================================
# 1. Mutex (Lock) — Exclusive Access
# ============================================
import threading

class BankAccount:
    def __init__(self, balance: float):
        self.balance = balance
        self._lock = threading.Lock()  # mutex
    
    def withdraw(self, amount: float, name: str):
        with self._lock:  # acquire → critical section → release
            if self.balance >= amount:
                self.balance -= amount
                print(f"  {name} withdrew ${amount}. Balance: ${self.balance}")
            else:
                print(f"  {name}: Insufficient funds (${self.balance} < ${amount})")


account = BankAccount(1000)
threads = [
    threading.Thread(target=account.withdraw, args=(800, "Alice")),
    threading.Thread(target=account.withdraw, args=(800, "Bob")),
]
for t in threads: t.start()
for t in threads: t.join()
print(f"  Final: ${account.balance}")

In [ ]:
# ============================================
# 2. Counting Semaphore — Connection Pool
# ============================================
import threading
import time

class ConnectionPool:
    def __init__(self, max_connections: int):
        self._semaphore = threading.Semaphore(max_connections)
        self._active = 0
        self._lock = threading.Lock()
    
    def acquire_connection(self, client_id: int):
        print(f"  Client {client_id}: waiting for connection...")
        self._semaphore.acquire()  # blocks if 0 available
        
        with self._lock:
            self._active += 1
        print(f"  Client {client_id}: connected! (active: {self._active})")
        
        time.sleep(0.3)  # simulate work
        
        with self._lock:
            self._active -= 1
        self._semaphore.release()
        print(f"  Client {client_id}: released. (active: {self._active})")


pool = ConnectionPool(max_connections=3)
threads = [threading.Thread(target=pool.acquire_connection, args=(i,)) for i in range(6)]
for t in threads: t.start()
for t in threads: t.join()
print("\n  All clients done!")

In [ ]:
# ============================================
# 3. Condition Variable — Wait/Notify
# ============================================
import threading

class BoundedBuffer:
    """Monitor-style bounded buffer with condition variables."""
    
    def __init__(self, capacity: int):
        self._buffer = []
        self._capacity = capacity
        self._lock = threading.Lock()
        self._not_full = threading.Condition(self._lock)
        self._not_empty = threading.Condition(self._lock)
    
    def produce(self, item):
        with self._not_full:
            while len(self._buffer) >= self._capacity:
                self._not_full.wait()  # sleep until space available
            self._buffer.append(item)
            print(f"  📦 Produced: {item} | Buffer: {self._buffer}")
            self._not_empty.notify()  # wake up consumer
    
    def consume(self):
        with self._not_empty:
            while not self._buffer:
                self._not_empty.wait()  # sleep until item available
            item = self._buffer.pop(0)
            print(f"  🛒 Consumed: {item} | Buffer: {self._buffer}")
            self._not_full.notify()  # wake up producer
            return item


buf = BoundedBuffer(capacity=3)

def producer():
    for i in range(5):
        buf.produce(f"item_{i}")

def consumer():
    for _ in range(5):
        buf.consume()

t1 = threading.Thread(target=producer)
t2 = threading.Thread(target=consumer)
t1.start(); t2.start()
t1.join(); t2.join()

## 📝 Interview Questions

| # | Question | Key Point |
|---|----------|------------|
| 1 | Mutex vs Semaphore? | Mutex = ownership + binary. Semaphore = no ownership + counting |
| 2 | Binary semaphore vs mutex? | Binary sem can be signaled by any thread. Mutex only by owner. Mutex has priority inheritance |
| 3 | What is a monitor? | High-level sync: encapsulates mutex + condition variables. Java synchronized blocks are monitors |
| 4 | Why `while` not `if` with condition vars? | Spurious wakeups! Always re-check the condition after waking up |
| 5 | Spinlock vs mutex? | Spinlock busy-waits (wastes CPU, but fast for short CS). Mutex sleeps (context switch overhead) |

---

## ┌──────────────────────────────────────────────────────────┐
## │              📌 KEY TAKEAWAYS                            │
## ├──────────────────────────────────────────────────────────┤
## │                                                          │
## │  • Mutex = owned binary lock (only owner unlocks)        │
## │  • Semaphore = counting signal (anyone can signal)       │
## │  • Condition var = wait until condition met (inside lock)│
## │  • Always use while-loop with condition variables        │
## │  • Python: Lock, Semaphore, Condition, Event             │
## │                                                          │
## └──────────────────────────────────────────────────────────┘

---

## ⏭️ Next Up: **Classic Sync Problems** — Producer-Consumer, Readers-Writers, Dining Philosophers